# 🌍 AQI Prediction using Multiple Linear Regression

## ⚠️Problem Statement

Air pollution is one of the major environmental issues affecting human health worldwide. 
The Air Quality Index (AQI) is a standardized measure used to indicate the level of air pollution.

Predicting AQI based on pollutant concentrations and weather conditions can help authorities and 
citizens take preventive measures and improve environmental monitoring.

## 🎯 Goal

To develop a machine learning model that accurately predicts the Air Quality Index (AQI) using pollutant concentrations and weather-related factors.

## 📌 Project Objectives

• 📊 Perform exploratory data analysis to understand the structure and patterns in the air quality dataset.

• 🧹 Clean and preprocess the dataset by handling missing values, outliers, and inconsistencies.

• 🔍 Analyze the relationship between pollutant levels, weather conditions, and AQI.

• 🤖 Train a machine learning model to predict AQI based on environmental features.

• 📈 Evaluate the model’s performance and extract insights about the key factors influencing air pollution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
sns.set_style("whitegrid")
plt.rcParams.update(plt.rcParamsDefault)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Load Dataset

In [ ]:
df=pd.read_csv("../data/aqi_dataset.csv")

In [ ]:
df.head()

## Dataset Overview

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Feature Selection
Removing unnecessary columns that are not required for model training.

In [ ]:
df=df.drop(columns=["datetime","date","city","station","aqi_category"])

In [ ]:
df["pm_ratio"] = df["pm25"] / (df["pm10"] + 1)

## Missingvalues Analysis

In [ ]:
df.isnull().sum()

The dataset does not contain significant missing values, indicating good data quality.  
Since missing values are minimal, they can either be ignored or handled using simple imputation techniques if required during preprocessing.

## Duplicate Records Check

In [ ]:
df.duplicated().sum()

## Outlier Detection

In [ ]:
sns.boxplot(data=df[["pm25","pm10","no2","so2","co","o3"]])
plt.show()



The boxplots show the presence of several outliers in pollutant concentration features such as PM25 and PM10.  
These outliers may represent extreme pollution events, which are common in real-world environmental datasets.

Instead of removing them directly, they should be handled carefully since they may contain important information about severe pollution conditions.

In [ ]:
columns= ["pm25","pm10","no2","so2","co","o3"]
for i in columns:
    Q1 = df[i].quantile(0.25)
    Q3 = df[i].quantile(0.75)
    IQR = Q3 - Q1
    minimum = Q1 - 1.5 * IQR
    maximum = Q3 + 1.5 * IQR
    df = df[(df[i] >= minimum) & (df[i] <= maximum)]

In [ ]:
sns.boxplot(data=df[["pm25","pm10","no2","so2","co","o3"]])
plt.show()

In [ ]:
df.shape

## Data Type Verification

In [ ]:
df.dtypes

## 🔎 Exploratory Data Analysis (EDA)

Exploratory Data Analysis helps in understanding the structure of the dataset and identifying patterns that influence AQI.

### Key Focus Areas

• 📉 Distribution of pollutant concentrations  
• 📊 Relationship between pollutants and AQI  
• 🌦️ Impact of weather conditions on air quality  
• 🕒 Variation of AQI across different months and hours  
• 🔗 Correlation between different environmental variables

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["aqi"], kde=True, bins=30)
plt.title("Distribution of AQI")
plt.xlabel("AQI")
plt.ylabel("Frequency")
plt.show()

In [ ]:
pollution_features = ["pm25","pm10","no2","so2","co","o3"]

plt.figure(figsize=(18,15))

i = 1
for col in pollution_features:
    plt.subplot(3, 2, i)   # 2 rows, 3 columns
    sns.scatterplot(x=df[col], y=df["aqi"])
    plt.title(f"{col} vs AQI")
    plt.xlabel(col)
    plt.ylabel("AQI")
    i += 1

plt.tight_layout()
plt.show()

• PM25 and PM10 show the strongest positive relationship with AQI, indicating particulate matter is the main contributor to air pollution.

• NO₂ and SO₂ also increase with AQI, suggesting emissions from vehicles and industrial activities affect air quality.

• CO levels rise along with AQI, showing that combustion-related pollution contributes to higher pollution levels.

• O₃ shows a weaker and more scattered relationship with AQI, indicating its impact on AQI is less consistent compared to other pollutants

In [ ]:
weather_features=["temperature","humidity","wind_speed","visibility"]
plt.figure(figsize=(18,8))
i=1
for columns in weather_features:
    plt.subplot(2,2,i)
    sns.scatterplot(x=df[columns],y=df["aqi"])
    plt.xlabel(columns)
    plt.ylabel("AQI")
    plt.title(f"Relationship between {columns} and AQI")
    i=i+1
plt.tight_layout()
plt.show()

• Temperature shows a weak relationship with AQI, indicating temperature alone does not strongly determine pollution levels.

• Humidity shows a slight effect on AQI, where higher humidity can sometimes correspond with increased pollution due to pollutant accumulation.

• Wind speed tends to reduce AQI, suggesting stronger winds help disperse pollutants and improve air quality.

• Visibility decreases as AQI increases, indicating poorer air quality leads to reduced atmospheric visibility.

## Time Features Vs AQI

In [ ]:
month_map = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
df["month_name"] = df["month"].map(month_map)

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(x=df["month_name"],y=df["aqi"],color="lightblue")
plt.xlabel("Month")
plt.ylabel("AQI")
plt.title(f"AQI variation across Month")
plt.tight_layout()
plt.show()

The boxplot shows that AQI is highest during winter months (October–January), indicating severe pollution. AQI gradually decreases towards summer and reaches the lowest levels during monsoon months (July–August). After September, pollution levels start increasing again. The presence of outliers suggests occasional extreme pollution events.

In [ ]:
plt.figure(figsize=(8,6))
sns.lineplot(x=df["hour"],y=df["aqi"],marker="o")
plt.xlabel("Hours")
plt.ylabel("AQI")
plt.title(f"AQI variation across Month")
plt.tight_layout()
plt.show()

The line plot shows that AQI gradually increases from early morning and reaches its peak around midday. After the afternoon, AQI begins to decrease slightly towards the evening and night. This indicates that pollution levels are relatively higher during the middle of the day compared to early morning and late night.


## Correlation Analysis



In [ ]:
correlation_matrix=df.corr(numeric_only=True)
plt.figure(figsize=(18,15))
sns.heatmap(correlation_matrix,annot=True,cmap="viridis",linewidth=0.5)
plt.title("Correlation Heatmap")
plt.show()

## Key Findings from Exploratory Data Analysis

PM25 and PM10 strongly influence AQI

Winter months show higher pollution

Weather variables influence pollutant dispersion

AQI varies across hours of the day

## 🤖 Machine Learning Model Development

In [ ]:
df = pd.get_dummies(df, columns=["season", "month_name","day_of_week"], drop_first=True)

In [ ]:
x=df.drop("aqi",axis=1)
y=df["aqi"]

### Train-Test Split

In [ ]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
x_train.shape,x_test.shape

In [ ]:
x_train.select_dtypes(include="object").columns

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [ ]:
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.fit_transform(x_test)

### Model Training – Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(x_train_scaled,y_train)

### Model Prediction

In [ ]:
y_predict=model.predict(x_test_scaled)

### Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
mae=mean_absolute_error(y_test,y_predict)
mse=mean_squared_error(y_test,y_predict)
rmse=np.sqrt(mse)
r2score=r2_score(y_test,y_predict)
print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R2SCORE:",r2score)


• MAE = 16.41
On average, the model prediction is about 16 AQI units away from the actual value.

• MSE = 497.54
This shows the average squared prediction error of the model. It indicates the overall magnitude of prediction errors during evaluation

• RMSE = 22.30
Some predictions may have larger errors around 22 AQI units.

• R² Score = 0.98
The model explains about 98% of the AQI pattern, which means it learned the relationship between features and AQI very well.


### Actual vs Predicted AQI

In [ ]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test, y=y_predict,s=6,color="#4FA3D1", alpha=0.8,edgecolor=None)
# perfect prediction line
plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()],color="red", linestyle="-",linewidth=2)
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Actual vs Predicted AQI")
plt.show()

• The graph compares actual AQI values and predicted AQI values from the model.

• Most of the points are close to the red reference line, which means the predicted values are very close to the actual AQI values.

• This shows that the model has learned the relationship between the features and AQI well.

• Only a few points are slightly away from the line, indicating small prediction errors.

• Overall, the graph shows that the model predictions follow the actual AQI trend closely and the model performs well.

### Residual Analysis

In [ ]:
residuals=y_test-y_predict
plt.figure(figsize=(6,4))
sns.histplot(residuals,kde=True)
plt.title("Residual Distribution")
plt.show()

• The residual plot shows the difference between actual AQI and predicted AQI values.

• Most of the residual values are centered around zero, which means the model predictions are generally close to the actual AQI values.

• The distribution looks approximately symmetric, indicating that the model does not consistently overpredict or underpredict AQI.

• Only a small number of residuals appear far from zero, suggesting few larger prediction errors.

• Overall, the residual distribution indicates that the model fits the data reasonably well and predictions are mostly accurate.

In [ ]:
coefficients = pd.DataFrame({"Feature": x.columns,"Coefficient": model.coef_})
# sort by importance
coefficients = coefficients.sort_values(by="Coefficient", key=abs, ascending=False)
# take top 10 most important
top_features = coefficients.head(10)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x="Coefficient",y="Feature",data=top_features,palette="viridis")
plt.title("Top Features Influencing AQI")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.show()

• PM10 has the highest influence on AQI, indicating that higher PM10 concentrations significantly increase air pollution levels.

• PM25 is the second most influential feature, showing that fine particulate matter strongly contributes to higher AQI values.

• Seasonal factors such as winter also affect AQI, suggesting that pollution levels tend to increase during colder months.

• Certain months like March and December show noticeable influence, indicating that air quality changes across different times of the year.

• Carbon monoxide (CO) also contributes to AQI, though its impact is smaller compared to particulate pollutants.

• Overall, the results show that particulate pollutants (PM10 and PM2.5) are the most important factors affecting air quality, while seasonal and temporal features provide additional influence.

## Conclusion

This project developed a machine learning model to predict the Air Quality Index (AQI) using pollutant concentrations, weather conditions, and temporal features. Exploratory Data Analysis revealed strong relationships between particulate pollutants such as PM10 and PM2.5 and AQI levels. Seasonal and monthly patterns also showed noticeable influence on air quality.

A Linear Regression model was trained using the processed dataset, and the model demonstrated strong predictive performance with an R² score of approximately 0.98. Evaluation metrics such as MAE and RMSE indicate that the model predictions are reasonably accurate and closely follow the actual AQI values.

Feature importance analysis highlights that particulate pollutants are the most significant contributors to AQI, while seasonal factors provide additional context to pollution patterns. Overall, the results show that machine learning techniques can effectively model and predict air quality levels based on environmental data.